# StudioAvatar – Tus avatares guardados (AVATARES 2026)

**Antes de empezar:** menú *Entorno de ejecución › Cambiar tipo de entorno › T4 GPU › Guardar*.

Siempre ejecuta primero el **Paso 0**. Después usa el botón que necesites:

| Botón | Qué hace | ¿GPU? |
|---|---|---|
| **1 · Crear / actualizar avatar** | Guarda tus fotos de referencia (o saca fotos de un video tuyo) | No |
| **2 · Crear looks** | Crea fotos tuyas con otro vestuario y las guarda en tu **banco de looks** | Sí |
| **3 · Voz del avatar** | Convierte un texto en audio con tu voz (Chatterbox, gratis) | Recomendada |
| **4 · Video que habla / canta** | Un look tuyo + un audio = video moviendo los labios | Sí |
| **5 · Ver mi banco** | Muestra todos tus looks, audios y videos guardados | No |

**La idea:** creas tus looks **una sola vez** y quedan guardados en tu Drive. Después solo los eliges para hacer videos, sin volver a generar el avatar.

### Tus carpetas en Google Drive
```
Fabrica de Videos/MIS AVATARES/AVATARES 2026/
└── <NOMBRE_AVATAR>/                 ← ej. EL_HYAKO_MAYOR
    ├── 01_fotos_referencia/         ← tus fotos reales (5 a 20, cara visible)
    ├── 02_looks/                    ← ⭐ TU BANCO DE AVATARES GUARDADOS
    │   ├── 01_STREET/  02_LUXURY/  03_HIPHOP/  04_FORMAL/
    │   └── 05_VACACIONES/  06_SPORT/  07_CASUAL/  08_FUTURISTA/ ...
    ├── 03_voz/muestra_voz.wav       ← 10 a 30 seg de tu voz limpia
    ├── 04_audios/                   ← audios creados con tu voz
    ├── 05_videos/                   ← videos de tu avatar hablando o cantando
    ├── 06_lora/                     ← (opcional) tu LoRA entrenado
    ├── avatar.json                  ← datos del avatar (no tocar)
    └── ficha.txt                    ← prompt maestro de tu identidad
```
**Conexión con StudioVoz:** el botón 3 puede mandar el audio a `StudioVoz/convertir` para pasarlo por TU modelo de voz (botón 3 de StudioVoz). El botón 4 puede usar los audios de `StudioVoz/conversiones`.

In [ ]:
#@title Paso 0 – Preparar (ejecútalo SIEMPRE primero) { display-mode: "form" }
import os, re, sys, glob, json, time, math, shutil, zipfile, subprocess, threading, html, base64, io, random
from google.colab import drive, files
from IPython.display import display, HTML

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

MYDRIVE = "/content/drive/MyDrive"
NOMBRE_RAIZ = "AVATARES 2026"
STUDIOVOZ = f"{MYDRIVE}/StudioVoz"
MOTORES = "/content/motores"
REPOS = f"{MOTORES}/repos"
REGISTRO = "/content/registro_studioavatar.txt"
EXT_IMG = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
EXT_AUDIO = (".mp3", ".wav", ".flac", ".m4a", ".ogg", ".aif", ".aiff", ".opus")
EXT_VIDEO = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
SUB = {"fotos": "01_fotos_referencia", "looks": "02_looks", "voz": "03_voz",
       "audios": "04_audios", "videos": "05_videos", "lora": "06_lora"}
ESTILOS = {
    "01_STREET": "wearing a black hoodie, red bomber jacket, black cap, silver chains and exclusive sneakers, on a city street at night with red neon lights",
    "02_LUXURY": "wearing a tailored black designer suit and a gold watch, in front of a luxury hotel at golden hour",
    "03_HIPHOP": "wearing an oversized black and red hoodie and heavy gold chains, on a rooftop at sunset with the city skyline",
    "04_FORMAL": "wearing a black tuxedo, white shirt and black bow tie, in an elegant dark office with soft window light",
    "05_VACACIONES": "wearing an open tropical print shirt, white linen shorts and sunglasses, on a tropical beach with palm trees, sunny day",
    "06_SPORT": "wearing a black and red tracksuit and running shoes, in a modern city park at sunrise",
    "07_CASUAL": "wearing blue jeans, a white t-shirt and a denim jacket, in a cozy coffee shop with natural light",
    "08_FUTURISTA": "wearing a black techwear jacket with red LED lines, in a futuristic neon city at night with rain, cinematic sci-fi",
}

class Error(Exception):
    pass

def buscar_raiz():
    """Busca la carpeta 'AVATARES 2026' en tu Drive (o la crea en Mi unidad)."""
    for c in (f"{MYDRIVE}/Fabrica de Videos/MIS AVATARES/{NOMBRE_RAIZ}", f"{MYDRIVE}/{NOMBRE_RAIZ}"):
        if os.path.isdir(c):
            return c
    for nivel in ("*", "*/*", "*/*/*"):
        r = [p for p in glob.glob(f"{MYDRIVE}/{nivel}/{NOMBRE_RAIZ}") if os.path.isdir(p)]
        if r:
            return r[0]
    c = f"{MYDRIVE}/Fabrica de Videos/MIS AVATARES/{NOMBRE_RAIZ}"
    os.makedirs(c, exist_ok=True)
    return c

RAIZ = buscar_raiz()

def limpio(nombre):
    n = re.sub(r"[^A-Za-z0-9_\-]", "_", str(nombre).strip())
    if not n.strip("_"):
        raise Error("Escribe un nombre (sin acentos ni símbolos).")
    return n

def ruta_drive(p):
    return "Mi unidad › " + str(p).replace(MYDRIVE + "/", "").replace("/", " › ")

def hms(seg):
    seg = int(seg)
    return f"{seg // 3600}:{seg % 3600 // 60:02d}:{seg % 60:02d}" if seg >= 3600 else f"{seg // 60:02d}:{seg % 60:02d}"

# ---------------- barra de progreso 0-100 ----------------
class Barra:
    """Barra 0-100 % con reloj. El reloj sigue corriendo mientras trabaja."""
    def __init__(self, titulo):
        self.titulo, self.t0 = titulo, time.time()
        self.pct, self.desde, self.hasta = 0.0, 0.0, 100.0
        self.fase, self.info, self.detalle = "Empezando...", "", ""
        self.estado, self.lento, self._pintado = "trabajando", None, 0
        self.h = display(HTML(""), display_id=True)
        self._fin = threading.Event()
        threading.Thread(target=self._reloj, daemon=True).start()
        self.pintar(True)

    def __enter__(self):
        return self

    def __exit__(self, tipo, valor, tb):
        if self.estado == "trabajando":
            if tipo is None:
                self.terminar(True, "Terminado")
            elif tipo is KeyboardInterrupt:
                self.terminar(False, "Detenido por ti")
            else:
                self.terminar(False, "Falló (mira el mensaje de abajo)")
        return False

    def _reloj(self):
        while not self._fin.wait(1):
            if self.lento:
                d, h, seg, t = self.lento
                self._poner(d + (h - d) * (1 - math.exp(-(time.time() - t) / seg)))
            self.pintar(True)

    def _poner(self, pct):
        self.pct = max(self.pct, min(100.0, pct))

    def etapa(self, fase, desde, hasta, estimado=None):
        self.fase, self.desde, self.hasta, self.info, self.detalle = fase, desde, hasta, "", ""
        self._poner(desde)
        self.lento = (desde, desde + (hasta - desde) * 0.95, estimado, time.time()) if estimado else None
        self.pintar(True)

    def avance(self, fraccion):
        self.lento = None
        self._poner(self.desde + (self.hasta - self.desde) * max(0.0, min(1.0, fraccion)))
        self.pintar()

    def terminar(self, ok, fase):
        self.estado = "ok" if ok else "error"
        self.fase, self.lento = fase, None
        if ok:
            self.pct, self.detalle = 100.0, ""
        self._fin.set()
        self.pintar(True)

    def pintar(self, forzar=False):
        if not forzar and time.time() - self._pintado < 0.5:
            return
        self._pintado = time.time()
        color = {"trabajando": "#1e88e5", "ok": "#43a047", "error": "#e53935"}[self.estado]
        icono = {"trabajando": "⏳", "ok": "✅", "error": "❌"}[self.estado]
        info = f" · <b>{html.escape(self.info)}</b>" if self.info else ""
        det = html.escape(self.detalle[:120])
        self.h.update(HTML(f"""
<div style="font-family:sans-serif;max-width:760px;margin:6px 0">
 <div style="display:flex;justify-content:space-between;font-size:15px">
  <span>{icono} <b>{html.escape(self.titulo)}</b> — {html.escape(self.fase)}</span>
  <b>{self.pct:.0f}%</b></div>
 <div style="background:rgba(128,128,128,.25);border-radius:8px;height:22px;overflow:hidden;margin:5px 0">
  <div style="width:{self.pct:.1f}%;height:100%;background:{color};transition:width .6s"></div></div>
 <div style="font-size:13px;opacity:.85">⏱ {hms(time.time() - self.t0)}{info}</div>
 <div style="font-size:11px;opacity:.55;font-family:monospace;white-space:nowrap;overflow:hidden">{det}</div>
</div>"""))

# ---------------- ejecutar comandos ----------------
def run(cmd, cwd=None, shell=False, al_leer=None, mostrar=True, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg", **(env_extra or {})}
    p = subprocess.Popen(cmd, cwd=cwd, shell=shell, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    cola = []
    with open(REGISTRO, "a") as reg:
        reg.write(f"\n$ {cmd}\n")
        for linea in p.stdout:
            reg.write(linea)
            if mostrar:
                print(linea, end="")
            cola.append(linea)
            if len(cola) > 300:
                del cola[:100]
            if al_leer:
                try:
                    al_leer(linea)
                except Exception:
                    pass
    p.wait()
    return p.returncode, cola

FALLOS = ("Traceback (most recent call last)", "out of memory", "CUDA error")
RUIDO = ("%|", "it/s", "s/it", "Warning", "warn(")

def paso(titulo, cmd, barra=None, lector=None, cwd="/content", env_extra=None):
    """Ejecuta un paso. Con barra: muestra progreso en vez de todo el texto."""
    if barra is None:
        print(f"\n===== {titulo} =====")
    def al_leer(linea):
        if barra is None:
            return
        r = lector(linea) if lector else None
        if r:
            barra.avance(r[0])
            if r[1]:
                barra.info = r[1]
        t = linea.strip()
        if t and not any(x in t for x in RUIDO):
            barra.detalle = t
    codigo, cola = run(cmd, cwd=cwd, al_leer=al_leer, mostrar=barra is None, env_extra=env_extra)
    texto = "".join(cola)
    if codigo != 0 or any(f in texto for f in FALLOS):
        if barra:
            barra.terminar(False, f"Falló: {titulo}")
            print("Últimas líneas del registro:\n" + "".join(cola[-40:]))
        if "out of memory" in texto:
            raise Error(f"Falló: {titulo}. La GPU se quedó sin memoria: baja CANTIDAD o usa CALIDAD rápida.")
        raise Error(f"Falló: {titulo}. Copia el texto de arriba y pégalo en el chat.")
    return cola

def lector_marcas(barra):
    """Lee las marcas que imprimen los programas: 'ETAPA texto' y 'AVANCE a/b texto'."""
    def leer(linea):
        t = linea.strip()
        if t.startswith("ETAPA "):
            barra.fase = t[6:]
            return None
        m = re.match(r"AVANCE (\d+)/(\d+)\s*(.*)", t)
        if m and int(m.group(2)):
            return int(m.group(1)) / int(m.group(2)), m.group(3) or None
    return leer

# ---------------- archivos: PC, link de Drive o carpeta de Drive ----------------
def descomprimir_todo(carpeta):
    while True:
        zips = glob.glob(carpeta + "/**/*.zip", recursive=True)
        if not zips:
            return
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(z[:-4])
            except zipfile.BadZipFile:
                print("ZIP dañado, lo salto:", os.path.basename(z))
            os.remove(z)

def bajar_link(texto, destino):
    """Descarga links de Google Drive (archivo, ZIP o carpeta) o links directos (ej. Hugging Face)."""
    try:
        import gdown
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", "gdown"], mostrar=False)
        import gdown
    links = [l for l in re.split(r"[\s,]+", str(texto).strip()) if l.startswith("http")]
    if not links:
        raise Error("Pega un link en el campo LINK (debe empezar con https://).")
    os.makedirs(destino, exist_ok=True)
    for l in links:
        l = l.replace("/blob/", "/resolve/")
        print("Descargando:", l)
        m = re.search(r"/folders/([\w-]+)|/d/([\w-]+)|[?&]id=([\w-]+)", l)
        try:
            if "drive.google.com" in l and "/folders/" in l:
                ok = gdown.download_folder(id=m.group(1), output=destino, quiet=False)
            elif ("drive.google.com" in l or "docs.google.com" in l) and m:
                ok = gdown.download(id=m.group(2) or m.group(3), output=destino + "/", quiet=False)
            else:
                ok = gdown.download(url=l, output=destino + "/", quiet=False)
        except Exception as e:
            print("  ", e)
            ok = None
        if not ok:
            raise Error("No pude descargar ese link. En Google Drive: clic derecho › Compartir › "
                        "Acceso general: 'Cualquier persona con el enlace', copia el link y pégalo otra vez.")
    descomprimir_todo(destino)

def nombre_libre(carpeta, nombre):
    base, ext = os.path.splitext(nombre)
    destino, n = f"{carpeta}/{nombre}", 1
    while os.path.exists(destino):
        destino, n = f"{carpeta}/{base}_{n}{ext}", n + 1
    return destino

def obtener_archivos(origen, destino, exts, link="", tipo="archivos"):
    """Trae archivos desde el PC o un link, y los deja en 'destino'. Devuelve sus rutas."""
    tmp = "/content/_entrada"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    if origen.startswith("Subir"):
        print("👉 Pulsa 'Choose Files' / 'Elegir archivos' (puedes elegir varios o un ZIP).")
        print("   No cambies de pestaña hasta que termine de subir. Si el botón sale gris, usa 'Link'.")
        for f in files.upload():
            shutil.move(f, f"{tmp}/{os.path.basename(f)}")
    else:
        bajar_link(link, tmp)
    descomprimir_todo(tmp)
    os.makedirs(destino, exist_ok=True)
    salida = []
    for raiz, _, nombres in os.walk(tmp):
        if "__MACOSX" in raiz:
            continue
        for f in sorted(nombres):
            if f.lower().endswith(exts) and not f.startswith("._"):
                d = nombre_libre(destino, f)
                shutil.move(os.path.join(raiz, f), d)
                salida.append(d)
    if not salida:
        raise Error(f"No encontré {tipo} en lo que subiste.")
    return salida

def duracion(ruta):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", ruta],
                       capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return 0.0

# ---------------- avatares ----------------
def lista(carpeta, exts, recursivo=False):
    if not os.path.isdir(carpeta):
        return []
    patron = carpeta + ("/**/*" if recursivo else "/*")
    return sorted((f for f in glob.glob(patron, recursive=recursivo) if f.lower().endswith(exts)),
                  key=os.path.getmtime)

def carpeta_avatar(nombre, crear=False):
    c = f"{RAIZ}/{limpio(nombre)}"
    if not os.path.isdir(c) and not crear:
        listar_avatares()
        raise Error(f"No existe el avatar '{nombre}'. Créalo con el botón 1 o escribe uno de la lista.")
    if crear:
        for s in SUB.values():
            os.makedirs(f"{c}/{s}", exist_ok=True)
    return c

def datos_avatar(c):
    try:
        return json.load(open(f"{c}/avatar.json"))
    except Exception:
        return {"nombre": os.path.basename(c), "genero": "Hombre"}

def listar_avatares():
    nombres = sorted(d for d in os.listdir(RAIZ) if os.path.isdir(f"{RAIZ}/{d}") and not d.startswith((".", "_")))
    if not nombres:
        print("Aún no tienes avatares. Empieza con el botón 1.")
        return
    print(f"Tus avatares ({ruta_drive(RAIZ)}):")
    for n in nombres:
        c = f"{RAIZ}/{n}"
        fotos = len(lista(f"{c}/{SUB['fotos']}", EXT_IMG))
        looks = len(lista(f"{c}/{SUB['looks']}", EXT_IMG, True))
        audios = len(lista(f"{c}/{SUB['audios']}", EXT_AUDIO))
        videos = len(lista(f"{c}/{SUB['videos']}", EXT_VIDEO))
        voz = "✅" if os.path.exists(f"{c}/{SUB['voz']}/muestra_voz.wav") else "—"
        print(f"  • {n}: {fotos} fotos | {looks} looks | voz {voz} | {audios} audios | {videos} videos")

def miniaturas(rutas, ancho=150, titulo=""):
    from PIL import Image, ImageOps
    partes = []
    for r in rutas:
        try:
            im = ImageOps.exif_transpose(Image.open(r)).convert("RGB")
            im.thumbnail((ancho * 2, ancho * 2))
            b = io.BytesIO(); im.save(b, "JPEG", quality=80)
            src = base64.b64encode(b.getvalue()).decode()
            partes.append(f'<div style="display:inline-block;margin:3px;text-align:center;font-size:10px;'
                          f'width:{ancho}px;overflow:hidden"><img src="data:image/jpeg;base64,{src}" '
                          f'style="width:{ancho}px;border-radius:6px"><br>{html.escape(os.path.basename(r))}</div>')
        except Exception:
            pass
    if partes:
        display(HTML((f"<h4 style='font-family:sans-serif;margin:8px 0 2px'>{html.escape(titulo)}</h4>" if titulo else "")
                     + "".join(partes)))

# ---------------- motores (cada uno en su propio entorno, para que no choquen) ----------------
UVX = [sys.executable, "-m", "uv"]

def py_motor(nombre):
    return f"{MOTORES}/{nombre}/bin/python"

def _uv(barra, titulo, py, paquetes, extra=()):
    paso(titulo, UVX + ["pip", "install", "--python", py, *extra, *paquetes], barra)

def _clonar(barra, url, carpeta):
    if not os.path.exists(f"{carpeta}/.git"):
        shutil.rmtree(carpeta, ignore_errors=True)
        paso("Descargar " + os.path.basename(carpeta), ["git", "clone", "-q", "--depth", "1", url, carpeta], barra)

def _bajar(barra, url, destino):
    if os.path.exists(destino) and os.path.getsize(destino) > 1000:
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    paso("Descargar " + os.path.basename(destino), ["wget", "-q", "-O", destino, url], barra)

def asegurar_motor(nombre, barra, desde=0, hasta=30):
    """Instala un motor la primera vez en esta sesión (imagen, voz o video)."""
    py, marca = py_motor(nombre), f"{MOTORES}/{nombre}/.ok"
    if os.path.exists(marca):
        return py
    t = lambda a, b: (desde + (hasta - desde) * a, desde + (hasta - desde) * b)
    os.environ["UV_HTTP_TIMEOUT"] = "300"
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], mostrar=False)
    os.makedirs(REPOS, exist_ok=True)
    version = {"imagen": "3.10", "voz": "3.11", "video": "3.10"}[nombre]
    barra.etapa(f"Instalando motor de {nombre} 1/3: preparando Python", *t(0, .08), estimado=20)
    shutil.rmtree(f"{MOTORES}/{nombre}", ignore_errors=True)
    paso("Crear entorno", UVX + ["venv", f"{MOTORES}/{nombre}", "--python", version], barra)
    barra.etapa(f"Instalando motor de {nombre} 2/3: librerías (lo más largo, ~4 min)", *t(.08, .8), estimado=150)
    if nombre == "imagen":
        _uv(barra, "Base", py, ["setuptools<70", "wheel", "cython", "numpy==1.26.4"])
        _uv(barra, "Torch", py, ["torch==2.4.1", "torchvision==0.19.1"])
        _uv(barra, "Librerías", py, ["diffusers==0.30.3", "transformers==4.44.2", "huggingface_hub==0.25.2",
                                     "accelerate==0.34.2", "peft==0.12.0", "safetensors==0.4.5",
                                     "onnxruntime==1.19.2", "onnx==1.16.2", "albumentations==1.4.3",
                                     "opencv-python-headless==4.10.0.84", "numpy==1.26.4", "Pillow"])
        _uv(barra, "InsightFace", py, ["insightface==0.7.3", "numpy==1.26.4"], ["--no-build-isolation"])
        barra.etapa("Instalando motor de imagen 3/3: PhotoMaker", *t(.8, 1), estimado=20)
        _clonar(barra, "https://github.com/TencentARC/PhotoMaker", f"{REPOS}/PhotoMaker")
        prueba = (f"import sys; sys.path.insert(0, '{REPOS}/PhotoMaker'); import torch, insightface, cv2; "
                  "from photomaker import PhotoMakerStableDiffusionXLPipeline, FaceAnalysis2; print('ok')")
    elif nombre == "voz":
        _uv(barra, "Chatterbox", py, ["chatterbox-tts==0.1.7"])
        barra.etapa("Instalando motor de voz 3/3: probando", *t(.8, 1), estimado=20)
        prueba = "from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print('ok')"
    else:  # video (SadTalker)
        _uv(barra, "Base", py, ["setuptools<70", "wheel", "cython", "numpy==1.23.4"])
        _uv(barra, "Torch", py, ["torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2", "numpy==1.23.4"])
        _uv(barra, "Librerías", py, [
            "torch==2.0.1", "torchvision==0.15.2", "numpy==1.23.4", "basicsr==1.4.2", "facexlib==0.3.0",
            "gfpgan==1.3.8", "face_alignment==1.3.5", "imageio==2.19.3", "imageio-ffmpeg==0.4.7",
            "librosa==0.9.2", "numba==0.56.4", "llvmlite==0.39.1", "resampy==0.3.1", "pydub==0.25.1",
            "scipy==1.10.1", "kornia==0.6.8", "tqdm", "yacs==0.1.8", "pyyaml", "joblib==1.1.0",
            "scikit-image==0.19.3", "safetensors", "av==11.0.0", "opencv-python==4.8.1.78"],
            ["--no-build-isolation"])
        barra.etapa("Instalando motor de video 3/3: SadTalker y sus modelos (~1.5 GB)", *t(.8, 1), estimado=90)
        st = f"{REPOS}/SadTalker"
        _clonar(barra, "https://github.com/OpenTalker/SadTalker", st)
        rel = "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc"
        for f in ("mapping_00109-model.pth.tar", "mapping_00229-model.pth.tar",
                  "SadTalker_V0.0.2_256.safetensors", "SadTalker_V0.0.2_512.safetensors"):
            _bajar(barra, f"{rel}/{f}", f"{st}/checkpoints/{f}")
        for url in ("https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
                    "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
                    "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
                    "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth"):
            _bajar(barra, url, f"{st}/gfpgan/weights/{url.rsplit('/', 1)[1]}")
        prueba = ("import torch, basicsr, gfpgan, facexlib, librosa, kornia, cv2, skimage, scipy, imageio, "
                  "safetensors, yaml, pydub; print('ok')")
    paso(f"Probar motor de {nombre}", [py, "-c", prueba], barra)
    open(marca, "w").write("ok")
    return py

# ---------------- programas de cada motor ----------------
SCRIPT_LOOKS = r'''
import json, os, sys, time
cfg = json.load(open(sys.argv[1]))
sys.path.insert(0, cfg["repo"])
import numpy as np, torch
from PIL import Image, ImageOps, PngImagePlugin
from diffusers import EulerDiscreteScheduler, AutoencoderKL
from huggingface_hub import hf_hub_download
from photomaker import PhotoMakerStableDiffusionXLPipeline, FaceAnalysis2, analyze_faces

print("ETAPA Revisando tus fotos de referencia", flush=True)
det = FaceAnalysis2(providers=["CPUExecutionProvider"], allowed_modules=["detection", "recognition"])
det.prepare(ctx_id=0, det_size=(640, 640))
refs, embeds = [], []
for ruta in cfg["fotos"]:
    try:
        im = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    except Exception:
        print("No pude abrir", os.path.basename(ruta), flush=True)
        continue
    im.thumbnail((1280, 1280))
    caras = analyze_faces(det, np.array(im)[:, :, ::-1])
    if not caras:
        print("Sin cara visible (la salto):", os.path.basename(ruta), flush=True)
        continue
    cara = max(caras, key=lambda c: (c["bbox"][2] - c["bbox"][0]) * (c["bbox"][3] - c["bbox"][1]))
    x1, y1, x2, y2 = cara["bbox"]
    cx, cy, lado = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1) * 2.2
    caja = (max(0, int(cx - lado / 2)), max(0, int(cy - lado / 2)),
            min(im.width, int(cx + lado / 2)), min(im.height, int(cy + lado / 2)))
    refs.append(im.crop(caja).resize((512, 512)))
    embeds.append(torch.from_numpy(cara["embedding"]))
    if len(refs) >= cfg["max_refs"]:
        break
if not refs:
    raise SystemExit("ERROR: ninguna foto tiene una cara visible. Sube fotos donde se vea bien tu cara.")
print(f"Fotos útiles: {len(refs)}", flush=True)
id_embeds = torch.stack(embeds)

print("ETAPA Cargando el modelo de imagen (la 1ª vez descarga ~7 GB)", flush=True)
dtype = torch.float16
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)
try:
    pipe = PhotoMakerStableDiffusionXLPipeline.from_pretrained(cfg["base"], vae=vae, torch_dtype=dtype, variant="fp16")
except Exception:
    pipe = PhotoMakerStableDiffusionXLPipeline.from_pretrained(cfg["base"], vae=vae, torch_dtype=dtype)
pipe.to("cuda")
ckpt = hf_hub_download(repo_id="TencentARC/PhotoMaker-V2", filename="photomaker-v2.bin", repo_type="model")
pipe.load_photomaker_adapter(os.path.dirname(ckpt), subfolder="", weight_name=os.path.basename(ckpt),
                             trigger_word="img", pm_version="v2")
pipe.fuse_lora()
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

n, pasos = cfg["cantidad"], cfg["pasos"]
total = n * pasos
os.makedirs(cfg["salida"], exist_ok=True)
for i in range(n):
    semilla = cfg["semilla"] + i
    print(f"ETAPA Creando look {i + 1} de {n}", flush=True)
    def avance(p, paso_actual, t, kw, i=i):
        print(f"AVANCE {i * pasos + paso_actual + 1}/{total} Look {i + 1} de {n}", flush=True)
        return kw
    img = pipe(cfg["prompt"], negative_prompt=cfg["negativo"], input_id_images=refs, id_embeds=id_embeds,
               num_images_per_prompt=1, num_inference_steps=pasos,
               start_merge_step=int(pasos * cfg["estilo"] / 100), guidance_scale=cfg["guia"],
               width=cfg["ancho"], height=cfg["alto"],
               generator=torch.Generator("cuda").manual_seed(semilla),
               callback_on_step_end=avance).images[0]
    meta = PngImagePlugin.PngInfo()
    meta.add_text("prompt", cfg["prompt"]); meta.add_text("semilla", str(semilla))
    ruta = os.path.join(cfg["salida"], f"{cfg['nombre']}_{time.strftime('%Y%m%d_%H%M%S')}_{semilla}.png")
    img.save(ruta, pnginfo=meta)
    print("GUARDADA", ruta, flush=True)
'''

SCRIPT_VOZ = r'''
import json, re, sys, torch, torchaudio as ta
cfg = json.load(open(sys.argv[1]))
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
print("ETAPA Cargando el modelo de voz (la 1ª vez descarga ~3 GB)", flush=True)
m = ChatterboxMultilingualTTS.from_pretrained(device="cuda" if torch.cuda.is_available() else "cpu")
frases = [f.strip() for f in re.split(r"(?<=[.!?…;:])\s+|\n+", cfg["texto"]) if f.strip()]
trozos, actual = [], ""
for f in frases:  # trozos de hasta ~250 letras (el modelo hace audios de máx. ~40 seg)
    while len(f) > 250:
        corte = f.rfind(" ", 0, 250)
        corte = corte if corte > 0 else 250
        if actual:
            trozos.append(actual); actual = ""
        trozos.append(f[:corte]); f = f[corte:].strip()
    if actual and len(actual) + len(f) + 1 > 250:
        trozos.append(actual); actual = f
    else:
        actual = (actual + " " + f).strip()
if actual:
    trozos.append(actual)
silencio = torch.zeros(1, int(m.sr * 0.25))
partes = []
for i, t in enumerate(trozos):
    print(f"ETAPA Hablando trozo {i + 1} de {len(trozos)}", flush=True)
    print(f"AVANCE {i}/{len(trozos)} Trozo {i + 1} de {len(trozos)}", flush=True)
    w = m.generate(t, language_id=cfg["idioma"], audio_prompt_path=cfg["muestra"],
                   exaggeration=cfg["emocion"], cfg_weight=cfg["ritmo"])
    partes += [w, silencio]
    print(f"AVANCE {i + 1}/{len(trozos)} Trozo {i + 1} de {len(trozos)}", flush=True)
ta.save(cfg["salida"], torch.cat(partes, dim=1), m.sr)
print("GUARDADO", cfg["salida"], flush=True)
'''

os.makedirs(f"{MOTORES}/scripts", exist_ok=True)
open(f"{MOTORES}/scripts/looks.py", "w").write(SCRIPT_LOOKS)
open(f"{MOTORES}/scripts/voz.py", "w").write(SCRIPT_VOZ)

try:
    HAY_GPU = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).returncode == 0
except Exception:
    HAY_GPU = False
print(f"Carpeta principal: {ruta_drive(RAIZ)}")
if HAY_GPU:
    print("GPU: ✅ activa")
else:
    print("GPU: ⚠️ NO HAY (o se acabó tu tiempo gratis de GPU de hoy).")
    print("   • Botones 1 y 5: funcionan igual.  • Botón 3 (voz): funciona, pero lento.")
    print("   • Botones 2 y 4: necesitan GPU. Vuelve más tarde o activa T4 en Entorno de ejecución › Cambiar tipo.")
print()
listar_avatares()
print("\n✅ Listo. Ahora usa el botón que necesites (1 a 5).")

## 1 · Crear / actualizar avatar
Guarda tus **fotos de referencia**. Se **acumulan**: puedes volver y agregar más.

- Lo ideal son **5 a 20 fotos** con la cara bien visible: de frente, de tres cuartos, con buena luz, sin filtros ni gafas oscuras.
- **¿Solo tienes un video?** Elige *Un video* y saco una foto por segundo (máx. 40). Después borra en Drive las borrosas.
- **Carpeta en mi Drive:** pon las fotos tú mismo en `AVATARES 2026/<avatar>/01_fotos_referencia` desde la app de Drive y ejecuta este botón para revisarlas.

In [ ]:
#@title 1 · Crear / actualizar avatar { display-mode: "form" }
NOMBRE_AVATAR = "EL_HYAKO_MAYOR" #@param {type:"string"}
GENERO = "Hombre" #@param ["Hombre", "Mujer"]
QUE_SUBES = "Fotos" #@param ["Fotos", "Un video (saco fotos de él)", "Nada (ya las puse en la carpeta de Drive)"]
ORIGEN = "Subir desde mi PC" #@param ["Subir desde mi PC", "Link de Google Drive"]
LINK_DRIVE = "" #@param {type:"string"}

if "limpio" not in globals():
    raise SystemExit("⚠️ Primero ejecuta el Paso 0. Colab se reinició y olvidó lo preparado.")
c = carpeta_avatar(NOMBRE_AVATAR, crear=True)
fotos_dir = f"{c}/{SUB['fotos']}"
datos = datos_avatar(c)
datos.update({"nombre": limpio(NOMBRE_AVATAR), "genero": GENERO})
datos.setdefault("creado", time.strftime("%Y-%m-%d %H:%M"))
json.dump(datos, open(f"{c}/avatar.json", "w"), ensure_ascii=False, indent=1)

if QUE_SUBES == "Fotos":
    nuevas = obtener_archivos(ORIGEN, fotos_dir, EXT_IMG, LINK_DRIVE, "fotos")
    print(f"✅ {len(nuevas)} fotos agregadas.")
elif QUE_SUBES.startswith("Un video"):
    videos = obtener_archivos(ORIGEN, "/content/_videos_ref", EXT_VIDEO, LINK_DRIVE, "videos")
    for v in videos:
        base = limpio(os.path.splitext(os.path.basename(v))[0])
        r = subprocess.run(["ffmpeg", "-nostdin", "-y", "-v", "error", "-i", v, "-vf", "fps=1,scale='min(1280,iw)':-2",
                            "-frames:v", "40", "-q:v", "2", f"{fotos_dir}/video_{base}_%03d.jpg"],
                           capture_output=True, text=True)
        print(("✅ Fotos sacadas de " if r.returncode == 0 else "❌ No pude leer ") + os.path.basename(v), r.stderr[:200])

ficha = f"{c}/ficha.txt"
if not os.path.exists(ficha):
    clase = "man" if GENERO == "Hombre" else "woman"
    open(ficha, "w").write(f"""AVATAR: {limpio(NOMBRE_AVATAR)}   ID: {limpio(NOMBRE_AVATAR)}_IDENTITY_001
CREADO: {datos['creado']}

PROMPT MAESTRO (identidad):
Ultra-realistic photo of the same real {clase} from the reference photos. Preserve exactly the face,
facial structure, eyes, nose, lips, jawline, skin tone, hairline, hair and body proportions.
Never beautify, age, de-age or replace with a generic model. Only clothing, accessories, pose,
expression, background, lighting and camera may change.
Professional DSLR editorial photography, natural skin texture, sharp facial focus, cinematic color.

REGLAS:
- Solo usar con fotos y voz propias (o con permiso escrito).
- Marcar como "contenido creado con IA" al publicar.
""")
todas = lista(fotos_dir, EXT_IMG)
print(f"\nAvatar '{limpio(NOMBRE_AVATAR)}': {len(todas)} fotos de referencia en {ruta_drive(fotos_dir)}")
if len(todas) < 3:
    print("⚠️ Con menos de 3 fotos el parecido baja. Lo ideal: 5 a 20.")
miniaturas(todas[-24:], 110, "Tus fotos de referencia")

## 2 · Crear looks (cambiar vestuario)
Crea fotos **de ti** con otro vestuario y las guarda en `02_looks/<ESTILO>`. Ese es tu **banco de avatares**: después solo eliges el look, sin volver a generarlo.

- Usa tus fotos de referencia con **PhotoMaker V2** sobre **RealVisXL** (modelos gratis y realistas).
- La primera vez instala el motor y descarga los modelos (~10 min). Las siguientes veces en la misma sesión es más rápido.
- Cada look tarda ~30–60 seg en una T4.
- **PARECIDO_VS_ESTILO:** más bajo = **más parecido a ti**; más alto = sigue más el vestuario y el fondo. Empieza con 20.
- Las fotos de **cuerpo completo** se parecen un poco menos (la cara sale más pequeña). Para el banco, empieza por **medio cuerpo**.
- Guarda solo los looks que te gusten. Los demás bórralos en Drive.

In [ ]:
#@title 2 · Crear looks { display-mode: "form" }
AVATAR = "EL_HYAKO_MAYOR" #@param {type:"string"}
ESTILO = "01_STREET" #@param ["01_STREET", "02_LUXURY", "03_HIPHOP", "04_FORMAL", "05_VACACIONES", "06_SPORT", "07_CASUAL", "08_FUTURISTA", "PERSONALIZADO"]
#@markdown Solo si eliges PERSONALIZADO (mejor en inglés). Ej.: *wearing a white linen suit, on a yacht at sunset*
DESCRIPCION_PERSONALIZADA = "" #@param {type:"string"}
NOMBRE_ESTILO_PERSONALIZADO = "09_MI_ESTILO" #@param {type:"string"}
ENCUADRE = "Medio cuerpo" #@param ["Medio cuerpo", "Retrato (pecho arriba)", "Cuerpo completo"]
FORMATO = "Vertical 9:16 (TikTok/Reels)" #@param ["Vertical 9:16 (TikTok/Reels)", "Cuadrado 1:1", "Horizontal 16:9 (YouTube)"]
CANTIDAD = 4 #@param {type:"slider", min:1, max:8, step:1}
PARECIDO_VS_ESTILO = 20 #@param {type:"slider", min:10, max:50, step:5}
#@markdown ---
#@markdown **Opciones avanzadas** (si no sabes qué son, déjalas así)
SEMILLA = -1 #@param {type:"integer"}
PASOS = 40 #@param {type:"integer"}
MAX_FOTOS_REFERENCIA = 10 #@param {type:"integer"}
MODELO_BASE = "SG161222/RealVisXL_V4.0" #@param {type:"string"}

if "limpio" not in globals():
    raise SystemExit("⚠️ Primero ejecuta el Paso 0. Colab se reinició y olvidó lo preparado.")
if not HAY_GPU:
    raise Error("Crear looks necesita GPU. Activa T4 en Entorno de ejecución › Cambiar tipo, o vuelve más tarde.")
c = carpeta_avatar(AVATAR)
fotos = lista(f"{c}/{SUB['fotos']}", EXT_IMG)
if not fotos:
    raise Error("Este avatar no tiene fotos de referencia. Usa primero el botón 1.")
clase = "man" if datos_avatar(c).get("genero", "Hombre") == "Hombre" else "woman"
if ESTILO == "PERSONALIZADO":
    if not DESCRIPCION_PERSONALIZADA.strip():
        raise Error("Escribe la ropa y el lugar en DESCRIPCION_PERSONALIZADA.")
    nombre_estilo, ropa = limpio(NOMBRE_ESTILO_PERSONALIZADO), DESCRIPCION_PERSONALIZADA.strip()
else:
    nombre_estilo, ropa = ESTILO, ESTILOS[ESTILO]
encuadre = {"Medio cuerpo": "half body shot", "Retrato (pecho arriba)": "close-up portrait, head and shoulders",
            "Cuerpo completo": "full body shot"}[ENCUADRE]
ancho, alto = {"Vertical 9:16 (TikTok/Reels)": (768, 1344), "Cuadrado 1:1": (1024, 1024),
               "Horizontal 16:9 (YouTube)": (1344, 768)}[FORMATO]
prompt = (f"professional photo of a {clase} img, {ropa}, {encuadre}, photorealistic, natural skin texture, "
          "sharp focus on face, DSLR, cinematic lighting, high detail")
negativo = ("(asymmetry, worst quality, low quality, illustration, 3d, 2d, painting, cartoons, sketch), "
            "deformed face, different person, extra fingers, deformed hands, plastic skin, blurry, watermark, text")
semilla = SEMILLA if SEMILLA >= 0 else random.randint(0, 2**31 - CANTIDAD - 1)
salida = f"{c}/{SUB['looks']}/{nombre_estilo}"
antes = set(lista(salida, EXT_IMG))
cfg = {"repo": f"{REPOS}/PhotoMaker", "fotos": fotos[::-1], "max_refs": max(1, MAX_FOTOS_REFERENCIA),
       "base": MODELO_BASE, "prompt": prompt, "negativo": negativo, "cantidad": CANTIDAD, "pasos": max(10, PASOS),
       "estilo": PARECIDO_VS_ESTILO, "guia": 5.0, "ancho": ancho, "alto": alto, "semilla": semilla,
       "salida": salida, "nombre": nombre_estilo}
json.dump(cfg, open("/content/_looks.json", "w"))
print("Prompt:", prompt)

with Barra(f"Looks {nombre_estilo} de {limpio(AVATAR)}") as barra:
    py = asegurar_motor("imagen", barra, 0, 30)
    barra.etapa("Preparando", 30, 45, estimado=240)
    lector = lector_marcas(barra)
    def lector_looks(linea):
        r = lector(linea)
        if r and barra.desde < 45:
            barra.etapa("Creando looks", 45, 100)
        return r
    paso("Crear looks", [py, f"{MOTORES}/scripts/looks.py", "/content/_looks.json"], barra, lector_looks,
         env_extra={"HF_HOME": "/content/hf"})
    barra.terminar(True, f"{len(set(lista(salida, EXT_IMG)) - antes)} looks guardados en tu Drive")

nuevos = [f for f in lista(salida, EXT_IMG) if f not in antes]
print(f"Guardados en {ruta_drive(salida)}  (semilla inicial: {semilla})")
miniaturas(nuevos, 220, f"Nuevos looks {nombre_estilo}")

## 3 · Voz del avatar (texto → tu voz)
Convierte un texto en audio que suena como tú, con **Chatterbox** (gratis, habla español).

- **La primera vez** sube una muestra de tu voz: 10 a 30 segundos, sola, sin música ni eco. Queda guardada en `03_voz/muestra_voz.wav` y ya no la vuelves a subir. Si tienes una acapela limpia de StudioVoz (`StudioVoz/limpios`), úsala con *Link de Google Drive*.
- Los audios quedan en `04_audios`.
- **ENVIAR_A_STUDIOVOZ:** copia el audio a `StudioVoz/convertir`. Después, en tu cuaderno StudioVoz, botón 3 con *Carpeta en mi Drive*, lo pasas por **tu modelo entrenado** para que suene todavía más a ti.

In [ ]:
#@title 3 · Voz del avatar { display-mode: "form" }
AVATAR = "EL_HYAKO_MAYOR" #@param {type:"string"}
TEXTO = "Bienvenidos a mi canal. Soy El Hyako Mayor y esto apenas comienza." #@param {type:"string"}
IDIOMA = "es" #@param ["es", "en", "pt", "fr", "it"]
MUESTRA = "Mi muestra guardada (03_voz)" #@param ["Mi muestra guardada (03_voz)", "Subir desde mi PC", "Link de Google Drive"]
LINK_MUESTRA = "" #@param {type:"string"}
NOMBRE_AUDIO = "saludo" #@param {type:"string"}
ENVIAR_A_STUDIOVOZ = False #@param {type:"boolean"}
#@markdown ---
#@markdown **Opciones avanzadas**
EMOCION = 0.5 #@param {type:"slider", min:0.25, max:2, step:0.05}
RITMO = 0.5 #@param {type:"slider", min:0.1, max:1, step:0.05}

if "limpio" not in globals():
    raise SystemExit("⚠️ Primero ejecuta el Paso 0. Colab se reinició y olvidó lo preparado.")
c = carpeta_avatar(AVATAR)
muestra = f"{c}/{SUB['voz']}/muestra_voz.wav"
if not MUESTRA.startswith("Mi muestra"):
    traidos = obtener_archivos(MUESTRA, "/content/_muestra", EXT_AUDIO + EXT_VIDEO, LINK_MUESTRA, "audios")
    r = subprocess.run(["ffmpeg", "-nostdin", "-y", "-v", "error", "-i", traidos[0], "-t", "30", "-ac", "1",
                        "-ar", "24000", muestra], capture_output=True, text=True)
    if r.returncode != 0:
        raise Error("No pude leer ese audio: " + r.stderr[:200])
    print(f"✅ Muestra de voz guardada en {ruta_drive(muestra)} (la próxima vez elige 'Mi muestra guardada').")
if not os.path.exists(muestra):
    raise Error("Todavía no tienes muestra de voz. Elige 'Subir desde mi PC' o 'Link de Google Drive' en MUESTRA.")
if not TEXTO.strip():
    raise Error("Escribe el TEXTO que quieres que diga tu avatar.")
salida = nombre_libre(f"{c}/{SUB['audios']}", f"{limpio(NOMBRE_AUDIO or 'audio')}_{time.strftime('%Y%m%d_%H%M')}.wav")
json.dump({"texto": TEXTO, "idioma": IDIOMA, "muestra": muestra, "emocion": EMOCION, "ritmo": RITMO,
           "salida": salida}, open("/content/_voz.json", "w"), ensure_ascii=False)

with Barra(f"Voz de {limpio(AVATAR)}") as barra:
    py = asegurar_motor("voz", barra, 0, 30)
    barra.etapa("Preparando", 30, 50, estimado=120)
    lector = lector_marcas(barra)
    def lector_voz(linea):
        r = lector(linea)
        if r and barra.desde < 50:
            barra.etapa("Hablando", 50, 100)
        return r
    paso("Crear voz", [py, f"{MOTORES}/scripts/voz.py", "/content/_voz.json"], barra, lector_voz,
         env_extra={"HF_HOME": "/content/hf"})
    barra.terminar(True, "Audio guardado en tu Drive")

print("✅", ruta_drive(salida))
if ENVIAR_A_STUDIOVOZ:
    os.makedirs(f"{STUDIOVOZ}/convertir", exist_ok=True)
    shutil.copy2(salida, f"{STUDIOVOZ}/convertir/{os.path.basename(salida)}")
    print("📨 Copiado a StudioVoz › convertir. Abre StudioVoz › Paso 0 › botón 3 con 'Carpeta en mi Drive'.")
from IPython.display import Audio
display(Audio(salida))

## 4 · Video que habla / canta
Junta **un look de tu banco + un audio** y crea un video de tu avatar moviendo los labios (**SadTalker**, gratis).

- **LOOK:** el más reciente del estilo elegido, o escribe el nombre exacto de un archivo en `NOMBRE_LOOK`.
- **AUDIO:** el último que creaste (botón 3), el último que convertiste en StudioVoz, o uno tuyo (tu canción, por ejemplo).
- **Para cantar:** separa la voz de la música primero (StudioVoz, botón 1 › *Canciones (solo sacar la voz)*). Usa esa voz aquí y en tu editor vuelve a poner la canción completa.
- Tarda ~1 a 3 min por cada 10 seg de audio en una T4. Los videos quedan en `05_videos`.
- **Para videos con movimiento de cuerpo y cámara**, lee la nota de abajo (Wan 2.2 / InfiniteTalk).

In [ ]:
#@title 4 · Video que habla / canta { display-mode: "form" }
AVATAR = "EL_HYAKO_MAYOR" #@param {type:"string"}
ESTILO_LOOK = "01_STREET" #@param ["01_STREET", "02_LUXURY", "03_HIPHOP", "04_FORMAL", "05_VACACIONES", "06_SPORT", "07_CASUAL", "08_FUTURISTA", "Cualquiera (el más reciente)"]
NOMBRE_LOOK = "" #@param {type:"string"}
AUDIO = "Último audio de mi avatar (04_audios)" #@param ["Último audio de mi avatar (04_audios)", "Último audio de StudioVoz (conversiones)", "Subir desde mi PC", "Link de Google Drive"]
LINK_AUDIO = "" #@param {type:"string"}
ENCUADRE = "Imagen completa (cuerpo y fondo)" #@param ["Imagen completa (cuerpo y fondo)", "Solo la cara (recorte)"]
CALIDAD = "Alta (512)" #@param ["Rápida (256)", "Alta (512)"]
MEJORAR_CARA = True #@param {type:"boolean"}
#@markdown ---
#@markdown **Opciones avanzadas**
MOVIMIENTO_CABEZA = 0 #@param {type:"slider", min:0, max:45, step:1}
EXPRESION = 1.0 #@param {type:"slider", min:0.5, max:2, step:0.1}
DESCARGAR = False #@param {type:"boolean"}

if "limpio" not in globals():
    raise SystemExit("⚠️ Primero ejecuta el Paso 0. Colab se reinició y olvidó lo preparado.")
if not HAY_GPU:
    raise Error("El video necesita GPU. Activa T4 en Entorno de ejecución › Cambiar tipo, o vuelve más tarde.")
c = carpeta_avatar(AVATAR)
looks_dir = f"{c}/{SUB['looks']}"
if NOMBRE_LOOK.strip():
    encontrados = [f for f in lista(looks_dir, EXT_IMG, True) if os.path.basename(f) == NOMBRE_LOOK.strip()]
    if not encontrados:
        raise Error(f"No encontré el look '{NOMBRE_LOOK}'. Mira los nombres con el botón 5.")
    look = encontrados[-1]
else:
    carpeta = looks_dir if ESTILO_LOOK.startswith("Cualquiera") else f"{looks_dir}/{ESTILO_LOOK}"
    disponibles = lista(carpeta, EXT_IMG, True)
    if not disponibles:
        raise Error(f"No hay looks en {ruta_drive(carpeta)}. Créalos con el botón 2.")
    look = disponibles[-1]

if AUDIO.startswith("Último audio de mi"):
    audios = lista(f"{c}/{SUB['audios']}", EXT_AUDIO)
    if not audios:
        raise Error("No hay audios en 04_audios. Créalos con el botón 3 o elige otra opción en AUDIO.")
    audio = audios[-1]
elif AUDIO.startswith("Último audio de StudioVoz"):
    audios = lista(f"{STUDIOVOZ}/conversiones", EXT_AUDIO)
    if not audios:
        raise Error("No hay audios en StudioVoz › conversiones.")
    audio = audios[-1]
else:
    audio = obtener_archivos(AUDIO, "/content/_audio_video", EXT_AUDIO + EXT_VIDEO, LINK_AUDIO, "audios")[0]
print("Look: ", ruta_drive(look) if look.startswith(MYDRIVE) else look)
print("Audio:", ruta_drive(audio) if audio.startswith(MYDRIVE) else os.path.basename(audio))

wav = "/content/_hablar.wav"
r = subprocess.run(["ffmpeg", "-nostdin", "-y", "-v", "error", "-i", audio, "-vn", "-ac", "1", "-ar", "16000", wav],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise Error("No pude leer el audio: " + r.stderr[:200])
seg = duracion(wav)
img = "/content/_look.png"
from PIL import Image, ImageOps
ImageOps.exif_transpose(Image.open(look)).convert("RGB").save(img)

st = f"{REPOS}/SadTalker"
res_dir = "/content/_sadtalker"
shutil.rmtree(res_dir, ignore_errors=True)
completa = ENCUADRE.startswith("Imagen completa")
tam = "512" if CALIDAD.startswith("Alta") else "256"
cmd = ["inference.py", "--driven_audio", wav, "--source_image", img, "--result_dir", res_dir,
       "--checkpoint_dir", "./checkpoints", "--size", tam, "--pose_style", str(MOVIMIENTO_CABEZA),
       "--expression_scale", str(EXPRESION), "--preprocess", "full" if completa else "crop"]
if completa:
    cmd.append("--still")
if MEJORAR_CARA:
    cmd += ["--enhancer", "gfpgan"]
estimado = 40 + seg * (6 if tam == "512" else 3) * (2.5 if MEJORAR_CARA else 1)

with Barra(f"Video de {limpio(AVATAR)} ({hms(seg)} de audio)") as barra:
    py = asegurar_motor("video", barra, 0, 30)
    cmd[0:0] = [py]
    barra.etapa("Animando tu avatar", 30, 100, estimado=estimado)
    paso("Crear video", cmd, barra, cwd=st)
    hechos = sorted(glob.glob(f"{res_dir}/*.mp4"), key=os.path.getmtime)
    if not hechos:
        raise Error("SadTalker no creó el video. Copia el texto de arriba y pégalo en el chat.")
    nombre = f"{os.path.splitext(os.path.basename(look))[0]}__{limpio(os.path.splitext(os.path.basename(audio))[0])}.mp4"
    final = nombre_libre(f"{c}/{SUB['videos']}", nombre)
    shutil.copy2(hechos[-1], final)
    barra.terminar(True, "Video guardado en tu Drive")

print("✅", ruta_drive(final))
from IPython.display import Video
display(Video(final, embed=True, width=360))
if DESCARGAR:
    files.download(final)

### 🎬 Videos con movimiento de cuerpo y cámara (Wan 2.2 / InfiniteTalk)
Los modelos de video más potentes (Wan 2.2, InfiniteTalk) **no caben en la GPU T4 gratis de Colab**. Úsalos así, con los looks de tu banco:

1. Descarga un look de `02_looks` desde tu Drive.
2. En **huggingface.co/spaces**, busca `Wan2.2` (imagen → video) o `InfiniteTalk` (foto + audio → video cantando con cuerpo). Tienen un límite gratis de minutos al día.
3. Sube tu look (y tu audio de `04_audios`), usa el prompt de abajo y guarda el resultado en `05_videos`.

```
the same man from the reference image, identical face. He walks slowly toward the camera
with confidence, slight smile, camera slowly moves back, cinematic lighting, realistic motion
```
**Kaggle** (kaggle.com, gratis con verificación por teléfono) da ~30 h de GPU por semana con más memoria, por si después quieres correr Wan 2.2 en un cuaderno.

## 5 · Ver mi banco de avatares

In [ ]:
#@title 5 · Ver mi banco { display-mode: "form" }
AVATAR = "EL_HYAKO_MAYOR" #@param {type:"string"}
MAX_POR_ESTILO = 12 #@param {type:"integer"}

if "limpio" not in globals():
    raise SystemExit("⚠️ Primero ejecuta el Paso 0. Colab se reinició y olvidó lo preparado.")
listar_avatares()
c = carpeta_avatar(AVATAR)
looks_dir = f"{c}/{SUB['looks']}"
estilos_hechos = sorted(d for d in os.listdir(looks_dir) if os.path.isdir(f"{looks_dir}/{d}")) if os.path.isdir(looks_dir) else []
if not estilos_hechos:
    print("\nTodavía no tienes looks. Créalos con el botón 2.")
for e in estilos_hechos:
    imgs = lista(f"{looks_dir}/{e}", EXT_IMG)
    if imgs:
        miniaturas(imgs[-MAX_POR_ESTILO:][::-1], 130, f"{e} ({len(imgs)} looks)")
for etiqueta, sub, exts in (("🎙️ Audios", "audios", EXT_AUDIO), ("🎬 Videos", "videos", EXT_VIDEO)):
    archivos = lista(f"{c}/{SUB[sub]}", exts)
    print(f"\n{etiqueta} ({len(archivos)}):")
    for f in archivos[-15:][::-1]:
        print("  •", os.path.basename(f))
print(f"\n📁 Todo está en {ruta_drive(c)}")